# 1. JAX Basics

The three JAX primitives that power every notebook in this tutorial:

| Primitive | What it does |
|-----------|-------------|
| `jit` | Compiles a Python function to XLA — pay the compilation cost once, run fast forever after |
| `vmap` | Vectorises a per-sample function over a batch with no explicit loop |
| `jax.random` | Explicit, functional PRNG — reproducible noise across runs and devices |

**Time:** ~30 minutes · **Prerequisites:** Python, NumPy. No JAX experience needed.

In [214]:
# @title Setup — run this cell first
# Uncomment on Colab:
# !pip install -q "jax[cuda12]" plotly jaxtyping

In [215]:
import time
from functools import partial

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from jaxtyping import Array, Float, PRNGKeyArray
from plotly.subplots import make_subplots

print(f"JAX {jax.__version__} · backend: {jax.default_backend()}")

JAX 0.10.1 · backend: cpu


---
## 1. JAX arrays

`jax.numpy` mirrors the NumPy API almost exactly — the same dtypes, indexing, and broadcasting rules. Two things are different:

1. **Device-first.** Arrays live on the accelerator (GPU/TPU) by default.
2. **Purely functional.** Operations have no side effects — they always return a new array and never mutate their inputs. This is what makes `jit` and `vmap` work safely.

In [216]:
a = jnp.array([1.0, 2.0, 3.0])
b = jnp.ones(3)

print("a + b  =", a + b)
print("a ** 2 =", a**2)
print("type   =", type(a))
print("device =", a.devices())
print("dtype  =", a.dtype)  # float32 by default

a + b  = [2. 3. 4.]
a ** 2 = [1. 4. 9.]
type   = <class 'jaxlib._jax.ArrayImpl'>
device = {CpuDevice(id=0)}
dtype  = float32


In [217]:
# NumPy allows in-place mutation
arr_np = np.array([1.0, 2.0, 3.0])
arr_np[0] = 99.0
print("NumPy:   ", arr_np)

# JAX does not
arr_jax = jnp.array([1.0, 2.0, 3.0])
try:
    arr_jax[0] = 99.0
except TypeError as e:
    print(f"JAX raises: {e}")

# The fix: .at[idx].set(value) returns a *new* array
arr_jax = arr_jax.at[0].set(99.0)
print("JAX fix: ", arr_jax)

NumPy:    [99.  2.  3.]
JAX raises: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html
JAX fix:  [99.  2.  3.]


### Exercise 1: Standardise an array

Implement `standardize(x)` so the output has **mean 0** and **standard deviation 1**.  
Use `jnp` operations only — no in-place mutation, no NumPy.

$$z = \frac{x - \mu}{\sigma}$$

where $\mu = \text{mean}(x)$ and $\sigma = \text{std}(x)$.

In [218]:
def standardize(x: Float[Array, " n"]) -> Float[Array, " n"]:
    """Return x with mean 0 and standard deviation 1."""
    # YOUR CODE HERE
    raise NotImplementedError

In [219]:
# @title 💡 Solution (open to reveal after trying to implement)


def standardize(x: Float[Array, " n"]) -> Float[Array, " n"]:
    return (x - x.mean()) / x.std()

In [220]:
# @title Test your function
rng = np.random.default_rng(0)
x = jnp.array(rng.normal(loc=3.0, scale=5.0, size=2000))
z = standardize(x)

assert jnp.abs(z.mean()) < 0.01, f"mean is {z.mean():.4f}, expected ≈ 0"
assert jnp.abs(z.std() - 1) < 0.01, f"std is {z.std():.4f}, expected ≈ 1"
print("✓ standardize works")

fig = go.Figure()
fig.add_trace(
    go.Histogram(x=np.array(x), name="original", opacity=0.7, marker_color="steelblue")
)
fig.add_trace(
    go.Histogram(x=np.array(z), name="standardised", opacity=0.7, marker_color="coral")
)
fig.update_layout(
    barmode="overlay",
    title="Original vs standardised distribution",
    xaxis_title="value",
    yaxis_title="count",
)
fig.show()

✓ standardize works


## 2. `jit`: compile once, run fast

### What is a JIT compiler?

Languages like C compile your code **ahead of time** — before it ever runs, the compiler reads the full program and emits an optimised binary. 

Python compiles source to bytecode which a virtual machine then executes. Modern CPython (3.11+) adds an adaptive specialising interpreter that rewrites hot operations based on observed types, and 3.13 ships an experimental JIT on top of that. This is fast enough for general-purpose code, but still leaves a lot on the table for tight numerical loops over large arrays.

A **Just-In-Time (JIT) compiler** waits until the code is actually called, observes what it does, and *then* compiles it — trading a one-off upfront cost for fast repeated execution. Python's JIT makes Python a faster language. JAX's JIT makes your maths run at hardware speed.

JAX's `jit` takes this further for array computing: rather than compiling Python bytecode, it extracts the full computation graph and hands it to **[XLA](https://openxla.org/xla)** (Accelerated Linear Algebra) — a domain-specific compiler originally built at Google to run ML workloads on CPUs, GPUs, and TPUs. XLA optimises the whole graph at once (fusing kernels, eliminating redundant memory copies, choosing efficient data layouts) and emits a single binary for the target device.

On the **first call**, JAX *traces* the function — it runs the Python body with abstract "tracer" values to record which operations happen and in what order, without needing concrete numbers. That recorded graph is handed to XLA. Every subsequent call with the same shapes and dtypes skips Python entirely and runs the compiled binary.

```
first call:       Python trace → XLA compile → execute   (slow)
subsequent calls:                              execute   (fast)
```

In [221]:
def heavy_computation(x):
    for _ in range(50):
        x = jnp.sin(x) + jnp.cos(x**2)
    return x


heavy_jit = jax.jit(heavy_computation)
x = jnp.linspace(0, 1, 512 * 512)

_ = heavy_jit(x).block_until_ready()  # warm up: first call compiles


def timeit_fn(fn, arg, n=30):
    start = time.perf_counter()
    for _ in range(n):
        fn(arg).block_until_ready()
    return (time.perf_counter() - start) / n * 1000  # ms per call


t_eager = timeit_fn(heavy_computation, x)
t_jit = timeit_fn(heavy_jit, x)

fig = px.bar(
    x=["eager", "jit"],
    y=[t_eager, t_jit],
    labels={"x": "", "y": "ms / call"},
    title=f"JIT speedup: {t_eager / t_jit:.1f}x",
    color=["eager", "jit"],
    color_discrete_map={"eager": "steelblue", "jit": "coral"},
)
fig.update_layout(showlegend=False)
fig.show()

### Gotcha 1: Python side effects run at *trace time*, not at *run time*

Inside `jit`, JAX records operations but does not execute Python eagerly. Side effects like `print` or appending to a list happen **once during the initial trace** and then never again.



In [222]:
call_log = []


@jax.jit
def f(x):
    call_log.append("called")  # Python side effect
    print("tracing!")  # also only happens during tracing
    return x * 2


for i in range(5):
    result = f(jnp.array(float(i)))
    print(f"Looping over f({i}) {result=}")

# Instead of 5 entries, call_log has only 1 entry:
# the Python body ran exactly once, during compilation
print(f"call_log has {len(call_log)} entr(ies): {call_log}")

tracing!
Looping over f(0) result=Array(0., dtype=float32, weak_type=True)
Looping over f(1) result=Array(2., dtype=float32, weak_type=True)
Looping over f(2) result=Array(4., dtype=float32, weak_type=True)
Looping over f(3) result=Array(6., dtype=float32, weak_type=True)
Looping over f(4) result=Array(8., dtype=float32, weak_type=True)
call_log has 1 entr(ies): ['called']


If you need to inspect a value at *runtime* inside a compiled function, JAX provides `jax.debug.print` — it inserts a side-effecting print into the compiled program itself, so it fires on every actual call


In [223]:
@jax.jit
def f(x):
    jax.debug.print("x = {x}", x=x)  # runs every call, not just at trace time
    return x * 2


for i in range(5):
    result = f(jnp.array(float(i)))
    print(f"Looping over f({i}) {result=}")

x = 0.0
Looping over f(0) result=Array(0., dtype=float32, weak_type=True)
x = 1.0
Looping over f(1) result=Array(2., dtype=float32, weak_type=True)
x = 2.0
Looping over f(2) result=Array(4., dtype=float32, weak_type=True)
x = 3.0
Looping over f(3) result=Array(6., dtype=float32, weak_type=True)
x = 4.0
Looping over f(4) result=Array(8., dtype=float32, weak_type=True)


### Gotcha 2: Traced values can't drive Python control flow

During tracing, array values are **abstract** — JAX knows their shape and dtype, but not their concrete numbers. `if x > 0` requires a concrete boolean, so JAX raises a `ConcretizationTypeError`.



In [224]:
@jax.jit
def bad_branch(x):
    if x > 0:  # ConcretizationTypeError: x is abstract at trace time
        return x * 2
    return x * -1


try:
    bad_branch(jnp.array(1.0))
except Exception as e:
    print(type(e).__name__)

TracerBoolConversionError


Use `jax.lax.cond` for data-dependent branches — JAX compiles *both* branches and selects one at runtime.

In [225]:
@jax.jit
def good_branch(x):
    return jax.lax.cond(
        x > 0,
        lambda x: x * 2,  # true branch
        lambda x: x * -1,  # false branch
        x,
    )


print(good_branch(jnp.array(3.0)))  # 6.0
print(good_branch(jnp.array(-3.0)))  # 3.0

6.0
3.0


### Gotcha 3: Shape-determining arguments must be `static`

During tracing, JAX works with abstract values — it knows shapes and dtypes, but not concrete numbers. This means anything that drives a Python-level loop (`range(n)`) or determines an array shape must be a concrete value known at compile time, not a traced one.

If you pass such an argument as a regular (traced) value, JAX raises an error.

In [226]:
@jax.jit
def denoise_n_steps(x, n_steps):
    for _ in range(n_steps):  # range() needs a concrete int
        x = x - 0.01 * x
    return x


try:
    denoise_n_steps(jnp.ones(4), 10)
except Exception as e:
    print(type(e).__name__, "--", e)

TracerIntegerConversionError -- The __index__() method was called on traced array with shape int32[]
The error occurred while tracing the function denoise_n_steps at /var/folders/xf/n_ydzkjn6yv26b8rtfk5tvf00000gn/T/ipykernel_45247/2232891355.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument n_steps.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerIntegerConversionError



These two gotchas can be addresed with these fixes:

**`static_argnums`** — a parameter to `jax.jit` that tells JAX which positional arguments should never be traced. JAX keeps them as concrete Python values and caches a separate compiled version for each unique combination of static values.

```python
# static_argnums takes a tuple of argument positions (0-indexed)
# Here position 1 (n_steps) is static, position 0 (x) is traced as normal
@partial(jax.jit, static_argnums=(1,))
def f(x, n_steps):
    ...
```

**`lax.fori_loop`** — a JAX-native loop that works with traced values and does not unroll.

In [227]:
# Fix 1: static_argnums — n_steps becomes a compile-time constant
@partial(jax.jit, static_argnums=(1,))
def denoise_static(x, n_steps):
    for _ in range(n_steps):  # n_steps is a concrete Python int
        x = x - 0.01 * x
    return x


# Fix 2: lax.fori_loop — n_steps can be a traced (runtime) value
@jax.jit
def denoise_dynamic(x, n_steps):
    return jax.lax.fori_loop(
        0,
        n_steps,
        lambda i, x: x - 0.01 * x,
        x,
    )


x_ref = jnp.ones(4)
print("static: ", denoise_static(x_ref, 10))
print("dynamic:", denoise_dynamic(x_ref, 10))
print("numpy:  ", np.ones(4) * (0.99**10))  # reference

static:  [0.9043821 0.9043821 0.9043821 0.9043821]
dynamic: [0.9043821 0.9043821 0.9043821 0.9043821]
numpy:   [0.90438208 0.90438208 0.90438208 0.90438208]


### Exercise 2: JIT + timing

1. JIT-compile `standardize` from Exercise 1 and assign the result to `standardize_jit`.
2. Run the validate cell to time both versions and see the speedup.
3. Answer in a comment: *does `standardize` need `static_argnums`?* Why or why not?

In [ ]:
# YOUR CODE HERE
standardize_jit = None
raise NotImplementedError
# YOUR ANSWER: Does standardize need static_argnums? Why?
# ...

NotImplementedError: 

In [ ]:
# --- validate ---
large_x = jnp.array(np.random.randn(1_000_000))

# Warm up: the first call traces and compiles the function.
# We run it once here so the compilation cost doesn't skew our
# timing measurements.
# We used `.block_until_ready()` toe ensure the computation
# is finished before we start timing since JAX dispatches
# operations asynchronously — Python moves on
#  before the GPU/CPU has actually finished.
_ = standardize_jit(large_x).block_until_ready()

t_eager = timeit_fn(standardize, large_x)
t_jit = timeit_fn(standardize_jit, large_x)

print(f"eager:   {t_eager:.2f} ms/call")
print(f"jit:     {t_jit:.2f} ms/call")
print(f"speedup: {t_eager / t_jit:.1f}x")

fig = px.bar(
    x=["eager", "jit"],
    y=[t_eager, t_jit],
    labels={"x": "", "y": "ms / call"},
    title=f"standardize speedup: {t_eager / t_jit:.1f}x",
    color=["eager", "jit"],
    color_discrete_map={"eager": "steelblue", "jit": "coral"},
)
fig.update_layout(showlegend=False)
fig.show()

---
## 3. `vmap`: vectorise without loops

`jax.vmap` transforms a function that operates on a **single sample** into one that operates on a **batch** — with no explicit Python loop. The result compiles to a single fused kernel: faster and simpler to read.

```python
# A function written for one sample
def normalize(x):           # x: (features,)
    return x / jnp.linalg.norm(x)

# vmap lifts it to work on a batch — no rewriting needed
normalize_batch = jax.vmap(normalize)

x      = jnp.ones(4)           # shape (4,)   — one sample
batch  = jnp.ones((8, 4))      # shape (8, 4) — 8 samples

normalize(x)            # shape (4,)   — works on one sample
normalize_batch(batch)  # shape (8, 4) — works on the whole batch
```

`in_axes` controls which argument is batched. `None` means the argument is shared across the whole batch:

```python
def scale(x, factor):           # x: one sample, factor: a scalar
    return x * factor

scale_batch = jax.vmap(scale, in_axes=(0, None))
# in_axes=(0, None) means:
#   axis 0 of x is the batch axis  → each row of x is one sample
#   factor is None                 → the same factor is used for every sample

scale_batch(batch, 2.0)         # shape (8, 4) — each row scaled by 2.0
```

In [229]:
def scale_sample(
    x: Float[Array, " features"], scale: float
) -> Float[Array, " features"]:
    """Scale a single 1-D sample."""
    return x * scale


# vmap over axis 0 of x; scale is shared (in_axes=None)
scale_batch = jax.vmap(scale_sample, in_axes=(0, None))

batch = jnp.ones((8, 4))  # 8 samples, each of length 4
out = scale_batch(batch, 2.0)
print("input shape: ", batch.shape)
print("output shape:", out.shape)
print(out)

input shape:  (8, 4)
output shape: (8, 4)
[[2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]
 [2. 2. 2. 2.]]


In [230]:
def loop_version(batch, scale):
    return jnp.stack([scale_sample(batch[i], scale) for i in range(batch.shape[0])])


scale_batch_jit = jax.jit(scale_batch)
loop_jit = jax.jit(loop_version)
big_batch = jnp.ones((256, 512))

_ = scale_batch_jit(big_batch, 2.0).block_until_ready()  # warm up
_ = loop_jit(big_batch, 2.0).block_until_ready()

t_loop = timeit_fn(lambda b: loop_jit(b, 2.0), big_batch, n=50)
t_vmap = timeit_fn(lambda b: scale_batch_jit(b, 2.0), big_batch, n=50)

fig = px.bar(
    x=["loop + jit", "vmap + jit"],
    y=[t_loop, t_vmap],
    labels={"x": "", "y": "ms / call"},
    title="vmap vs Python loop (both JIT-compiled)",
    color=["loop + jit", "vmap + jit"],
    color_discrete_map={"loop + jit": "steelblue", "vmap + jit": "coral"},
)
fig.update_layout(showlegend=False)
fig.show()

### Exercise 3: linear interpolation across a batch

The forward diffusion process blends a clean image with noise at each timestep. The building block is **linear interpolation**:

$$\text{lerp}(x_0, x_1, t) = (1 - t)\, x_0 + t\, x_1$$

1. Implement `lerp` for a **single pair** of 1-D vectors.
2. Use `vmap` to create `lerp_batch` that operates on two batches of vectors with a **shared** `t`.

*Hint:* `x0` and `x1` are batched, `t` is the same for every sample — think about what `in_axes` should be.

In [231]:
def lerp(
    x0: Float[Array, " n"],
    x1: Float[Array, " n"],
    t: float,
) -> Float[Array, " n"]:
    """Linearly interpolate between x0 and x1 at position t."""
    # YOUR CODE HERE
    raise NotImplementedError


# YOUR CODE HERE: define lerp_batch using vmap
# x0 and x1 are batched (one vector per sample), t is shared across the whole batch
lerp_batch = None
raise NotImplementedError

NotImplementedError: 

In [ ]:
# --- validate ---
rng = np.random.default_rng(0)
x0 = jnp.array(rng.normal(size=(16, 64)))
x1 = jnp.array(rng.normal(size=(16, 64)))

# single-pair checks
assert jnp.allclose(lerp(x0[0], x1[0], 0.0), x0[0]), "t=0 should return x0"
assert jnp.allclose(lerp(x0[0], x1[0], 1.0), x1[0]), "t=1 should return x1"
assert jnp.allclose(lerp(x0[0], x1[0], 0.5), 0.5 * x0[0] + 0.5 * x1[0]), (
    "t=0.5 should be midpoint"
)

# batched version
out = lerp_batch(x0, x1, 0.3)
assert out.shape == x0.shape, f"shape mismatch: {out.shape}"
print(f"✓ lerp_batch works — output shape: {out.shape}")

# Visualise lerp at t=0, 0.25, 0.5, 0.75, 1.0 for the first sample
fig = go.Figure()
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    fig.add_trace(
        go.Scatter(y=np.array(lerp(x0[0], x1[0], t)), name=f"t={t}", mode="lines")
    )
fig.update_layout(
    title="Lerp between x0 and x1 at different t values",
    xaxis_title="dimension",
    yaxis_title="value",
)
fig.show()

---
## 4. `jax.random`: explicit, functional PRNG

NumPy uses **global mutable state** for randomness:

```python
np.random.seed(0)    # sets global state
np.random.randn(3)   # reads and advances global state
```

This breaks under `jit` (state mutation is a side effect) and under parallelism (threads race on the same state).

JAX uses an **explicit, immutable key** instead. Every random call takes a key and returns a result — the key is never mutated. To get a fresh independent sample, you *split* the key first.

In [ ]:
# Same key always produces the same output
key = jr.PRNGKey(0)
print(jr.normal(key, shape=(3,)))
print(jr.normal(key, shape=(3,)))  # identical -- key did not change

In [ ]:
# Wrong: reusing the same key produces correlated (identical) samples
key = jr.PRNGKey(0)
samples_wrong = jnp.stack([jr.normal(key, (4,)) for _ in range(4)])
print("Wrong (reused key) -- every row is identical:")
print(samples_wrong)

In [ ]:
# Right: split the key to get independent subkeys
key = jr.PRNGKey(0)
keys = jr.split(key, num=4)  # shape (4, 2): four independent subkeys
samples_right = jnp.stack([jr.normal(k, (4,)) for k in keys])
print("Right (split keys) -- all rows differ:")
print(samples_right)

fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Reused key (wrong)", "Split keys (right)"]
)
for col, data in enumerate([samples_wrong, samples_right], start=1):
    for i, row in enumerate(np.array(data)):
        fig.add_trace(
            go.Histogram(x=row, name=f"sample {i}", showlegend=(col == 1)),
            row=1,
            col=col,
        )
fig.update_layout(
    barmode="overlay", title="Key reuse (rows collapse) vs key splitting (independent)"
)
fig.show()

The standard idiom for sequential key consumption:

```python
key, subkey = jr.split(key)            # consume one subkey, keep the rest in key
noise = jr.normal(subkey, shape)

key, subkey = jr.split(key)            # repeat as needed
more_noise = jr.normal(subkey, shape)
```

For a whole batch at once:

```python
keys = jr.split(key, num=batch_size)   # shape (batch_size, 2)
```

### Exercise 4: per-sample noise

Implement `sample_noise_batch(key, batch_size, dim)` returning a `(batch_size, dim)` array of **independent** standard-normal samples — one fresh subkey per sample.  
Do **not** reuse the same key.

In [ ]:
def sample_noise_batch(
    key: PRNGKeyArray, batch_size: int, dim: int
) -> Float[Array, "batch_size dim"]:
    """Return (batch_size, dim) independent standard-normal samples."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# @title 💡 Solution (open to reveal after trying to implement)


def sample_noise_batch(
    key: PRNGKeyArray, batch_size: int, dim: int
) -> Float[Array, "batch_size dim"]:
    keys = jr.split(key, num=batch_size)
    return jax.vmap(lambda k: jr.normal(k, shape=(dim,)))(keys)

In [ ]:
# --- validate ---
key = jr.PRNGKey(7)
noise = sample_noise_batch(key, batch_size=1000, dim=4)

assert noise.shape == (1000, 4), f"shape {noise.shape} != (1000, 4)"
assert not jnp.allclose(noise[0], noise[1]), (
    "rows are identical — did you reuse the key?"
)
assert jnp.abs(noise.mean()) < 0.1, "overall mean not near 0"
print("✓ sample_noise_batch works")

fig = px.histogram(
    x=np.array(noise).flatten(),
    nbins=60,
    title="Distribution of all noise samples (should look Gaussian)",
    labels={"x": "value", "y": "count"},
    color_discrete_sequence=["coral"],
)
fig.show()

---
## 5. Putting it all together

Let's combine `jit`, `vmap`, and `jax.random` into a **batched forward diffusion step** — the operation you will build properly in notebook 02.

Given a batch of images and a noise level `sigma`, we add independent Gaussian noise to each image. The pattern:

1. Write the function for **one image** — clean and readable
2. `vmap` it over the batch axis — efficient, no loop
3. `jit` the vmapped version — fast
4. Use properly split keys — correct

In [280]:
def add_noise_single(
    image: Float[Array, "h w"],
    key: PRNGKeyArray,
    sigma: float,
) -> Float[Array, "h w"]:
    """Add Gaussian noise to a single image."""
    noise = jr.normal(key, shape=image.shape)
    return image + sigma * noise


# Each image gets its own key; sigma is shared across the batch
add_noise_batch = jax.vmap(add_noise_single, in_axes=(0, 0, None))
add_noise_batch_jit = jax.jit(add_noise_batch)

In [283]:
batch_size = 8
key = jr.PRNGKey(42)

# Structured synthetic image: diagonal gradient, same for every item in the batch
x = jnp.linspace(0, 1, 32)
gradient = (x[None, :] + x[:, None]) / 2  # shape (32, 32), values in [0, 1]
images = jnp.stack([gradient] * batch_size)  # (8, 32, 32)

key, noise_key = jr.split(key)
noise_keys = jr.split(noise_key, batch_size)  # one independent key per image

noisy_images = add_noise_batch_jit(images, noise_keys, 0.4)


def make_image_grid(arrays, n_cols):
    """Tile a list of 2-D arrays into a single image grid."""
    rows = [
        np.concatenate(np.array(arrays[i : i + n_cols]), axis=1)
        for i in range(0, len(arrays), n_cols)
    ]
    return np.concatenate(rows, axis=0)


grid = make_image_grid(
    list(np.array(images)) + list(np.array(noisy_images)),
    n_cols=batch_size,
)

fig = px.imshow(
    grid,
    color_continuous_scale="gray",
    title="Top row: clean images  |  Bottom row: noisy (sigma=0.4)",
    zmin=0.0,
    zmax=1.0,
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

---
## What's next?

In **02 — Forward Process** you will replace the fixed `sigma` with a proper noise schedule (the $\beta_t$ sequence), and derive the closed-form formula that lets you jump to *any* noise level in a single step — no iterating through all intermediate timesteps required.